<a href="https://colab.research.google.com/github/Bilal-data/online-retail-analysis/blob/main/online_retail_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!wget -q -O retail.zip "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
!unzip -o retail.zip
!ls -lh

Archive:  retail.zip
 extracting: Online Retail.xlsx      
total 46M
-rwx------ 1 root root  23M May 22  2023 'Online Retail.xlsx'
-rw-r--r-- 1 root root  23M Nov 20 09:19  retail.zip
drwxr-xr-x 1 root root 4.0K Nov 18 15:58  sample_data


In [25]:
import pandas as pd

df = pd.read_excel("Online Retail.xlsx")
print(df.shape)
df.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [26]:
df.isna().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [27]:
(df["Quantity"] < 0).sum()

np.int64(10624)

In [28]:
df[df["Quantity"] < 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [29]:
neg = df[df["Quantity"] < 0]
neg["InvoiceNo"].astype(str).str.startswith("C").sum()

np.int64(9288)

In [30]:
not_c = neg[~neg["InvoiceNo"].astype(str).str.startswith("C")]
print(len(not_c))
not_c.head()

1336


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom


In [31]:
sales = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
print(len(sales))
print(len(df) - len(sales))

530104
11805


In [32]:
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["Revenue"].sum()

np.float64(10666684.544)

In [33]:
by_country = sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
(by_country / by_country.sum() * 100).round(1).head(5)

,Revenue
Country,
United Kingdom,84.6
Netherlands,2.7
EIRE,2.7
Germany,2.1
France,2.0


In [34]:
monthly = sales.groupby(sales["InvoiceDate"].dt.to_period("M"))["Revenue"].sum().round(0)
monthly

,Revenue
InvoiceDate,
2010-12,823746.0
2011-01,691365.0
2011-02,523632.0
2011-03,717639.0
2011-04,537809.0
2011-05,770536.0
2011-06,761740.0
2011-07,719221.0
2011-08,759138.0


In [35]:
sales["InvoiceDate"].max()

Timestamp('2011-12-09 12:50:00')

In [36]:
d10 = sales[(sales["InvoiceDate"] >= "2010-12-01") & (sales["InvoiceDate"] < "2010-12-10")]["Revenue"].sum()
d11 = sales[(sales["InvoiceDate"] >= "2011-12-01") & (sales["InvoiceDate"] < "2011-12-10")]["Revenue"].sum()
print(round(d10), round(d11))

438853 638793


In [37]:
p10 = (sales["InvoiceDate"] >= "2010-12-01") & (sales["InvoiceDate"] < "2010-12-10")
p11 = (sales["InvoiceDate"] >= "2011-12-01") & (sales["InvoiceDate"] < "2011-12-10")
print(sales[p10]["InvoiceNo"].nunique(), sales[p11]["InvoiceNo"].nunique())

834 819


In [38]:
inv10 = sales[p10].groupby("InvoiceNo")["Revenue"].sum().sort_values(ascending=False)
inv11 = sales[p11].groupby("InvoiceNo")["Revenue"].sum().sort_values(ascending=False)
print(inv10.head(3).round(0).tolist())
print(inv11.head(3).round(0).tolist())

[15885.0, 13541.0, 11540.0]
[168470.0, 10902.0, 10364.0]


In [39]:
big = sales["Revenue"].idxmax()
sales.loc[[big], ["InvoiceNo", "Description", "Quantity", "UnitPrice", "InvoiceDate", "Country"]]

,InvoiceNo,Description,Quantity,UnitPrice,InvoiceDate,Country
540421,581483,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,2011-12-09 09:15:00,United Kingdom


In [40]:
df[df["Quantity"].abs() == 80995][["InvoiceNo", "Description", "Quantity", "UnitPrice", "InvoiceDate"]]

,InvoiceNo,Description,Quantity,UnitPrice,InvoiceDate
540421,581483,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,2011-12-09 09:15:00
540422,C581484,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,2011-12-09 09:27:00


In [41]:
net = df[df["UnitPrice"] > 0].copy()
net["Revenue"] = net["Quantity"] * net["UnitPrice"]
n10 = net[(net["InvoiceDate"] >= "2010-12-01") & (net["InvoiceDate"] < "2010-12-10")]["Revenue"].sum()
n11 = net[(net["InvoiceDate"] >= "2011-12-01") & (net["InvoiceDate"] < "2011-12-10")]["Revenue"].sum()
print(round(n10), round(n11), round((n11 / n10 - 1) * 100, 1))

377488 433668 14.9


In [42]:
monthly_net = net.groupby(net["InvoiceDate"].dt.to_period("M"))["Revenue"].sum().round(0)
monthly_net

,Revenue
InvoiceDate,
2010-12,748957.0
2011-01,560000.0
2011-02,498063.0
2011-03,683267.0
2011-04,493207.0
2011-05,723334.0
2011-06,691123.0
2011-07,681300.0
2011-08,704805.0
